# 10 — Evaluation and Reporting

Comprehensive end-to-end evaluation across all trained models (classical and neural), with consistent test splits, per-class metrics, ROC/PR curves, confusion matrices, error analysis, ablations summary, and a consolidated comparison table for deployment recommendation.

In [6]:
# Imports and setup
from pathlib import Path
import numpy as np, pandas as pd, yaml, json, joblib, time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix, precision_recall_curve,
                             average_precision_score, roc_auc_score, roc_curve, accuracy_score, f1_score)
import tensorflow as tf
import tensorflow.lite as tflite
from tensorflow import keras
warnings_filter = __import__('warnings'); warnings_filter.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS_DIR = PROJECT_ROOT / 'results' / 'metrics'
FIG_DIR = PROJECT_ROOT / 'results' / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models'
NEURAL_DIR = PROJECT_ROOT / 'data' / 'processed' / 'neural_features'
CLASSICAL_DIR = PROJECT_ROOT / 'data' / 'processed' / 'classical_features'
for d in [RESULTS_DIR, FIG_DIR]: d.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)


Project root: /Users/harryirving/Development/projects/ai-ml/BikeAIv4


## Load data and models

In [7]:
# Load classical features and encoder
df_cls = pd.read_csv(CLASSICAL_DIR / 'classical_features.csv')
label_col = 'label'; path_col = 'path' if 'path' in df_cls.columns else 'segment_path'
le = joblib.load(MODELS_DIR / 'classical' / 'label_encoder.pkl')
y_cls = le.transform(df_cls[label_col].values)
X_cls = df_cls.drop(columns=[label_col, path_col] if path_col in df_cls.columns else [label_col]).values
scaler = joblib.load(MODELS_DIR / 'classical' / 'scaler.pkl')
X_cls = scaler.transform(X_cls)
classes = le.classes_; print('Classes:', classes)

# Classical models
models_classical = {}
for name in ['svm_tuned', 'rf_tuned', 'knn_tuned', 'xgb_tuned']:
    p = MODELS_DIR / 'classical' / f'{name}.pkl'
    if p.exists(): models_classical[name] = joblib.load(p)
print('Loaded classical:', list(models_classical.keys()))

# Skip neural arrays and CNN for now (Keras 2/3 incompatibility)
print('\nSkipping Custom CNN (Keras version mismatch—will re-run Notebook 07 with TF 2.14)')
cnn_best = None

# Load YAMNet head (should work if saved as .h5)
yam_head_path = MODELS_DIR / 'neural' / 'yamnet_head_best.h5'  # Adjust if you used .keras
try:
    yam_head = tf.keras.models.load_model(str(yam_head_path), compile=False) if yam_head_path.exists() else None
    print('Loaded YAMNet head:', yam_head is not None)
except Exception as e:
    print(f'YAMNet load failed: {e}')
    yam_head = None

print('Loaded models:', {'classical': len(models_classical), 'cnn': False, 'yamnet': yam_head is not None})


Classes: ['angle_grinder' 'background' 'tools']
Loaded classical: ['svm_tuned', 'knn_tuned']

Skipping Custom CNN (Keras version mismatch—will re-run Notebook 07 with TF 2.14)
Loaded YAMNet head: False
Loaded models: {'classical': 2, 'cnn': False, 'yamnet': False}


## Consistent test split

In [8]:
# Recreate consistent splits (file-level if available via manifest)
from sklearn.model_selection import train_test_split
manifest_path = RESULTS_DIR / 'preprocessing_manifest.csv'
if manifest_path.exists() and len(df_cls) == len(pd.read_csv(manifest_path)):
    man = pd.read_csv(manifest_path)
    stems = man['original_path'].apply(lambda p: Path(str(p)).stem)
    uniq = pd.DataFrame({'stem': stems, 'y': y_cls}).drop_duplicates()
    tr, te = train_test_split(uniq, test_size=0.2, stratify=uniq['y'], random_state=42)
    tr_mask = stems.isin(set(tr['stem']))
    te_mask = stems.isin(set(te['stem']))
else:
    # Fallback to sample-level splits
    idx = np.arange(len(y_cls))
    _, te_idx, _, _ = train_test_split(idx, y_cls, test_size=0.2, stratify=y_cls, random_state=42)
    te_mask = np.zeros_like(y_cls, dtype=bool); te_mask[te_idx]=True
    tr_mask = ~te_mask
# Test set for classical
X_test_cls, y_test = X_cls[te_mask], y_cls[te_mask]
print('Test size:', X_test_cls.shape[0])
# Neural test set alignment if shapes match
if X_neu is not None and len(X_neu) == len(y_cls):
    X_test_neu = X_neu[te_mask]
else:
    X_test_neu = None


Test size: 3809


## Evaluate classical models

In [9]:
results = []
for name, model in models_classical.items():
    t0 = time.time(); y_pred = model.predict(X_test_cls); inf = (time.time()-t0)/len(y_test)*1000
    acc = accuracy_score(y_test, y_pred); f1 = f1_score(y_test, y_pred, average='macro')
    rep = classification_report(y_test, y_pred, target_names=classes, output_dict=True)
    cm = confusion_matrix(y_test, y_pred)
    # Save CM
    plt.figure(figsize=(5,4)); sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(f'CM — {name}'); plt.tight_layout(); plt.savefig(FIG_DIR / f'cm_{name}.png', dpi=150); plt.close()
    results.append({'model': name, 'type': 'classical', 'accuracy': acc, 'f1_macro': f1, 'inf_ms': inf, 'report': rep})
df_cls_res = pd.DataFrame([{k:v for k,v in r.items() if k!='report'} for r in results if r['type']=='classical'])
df_cls_res


,model,type,accuracy,f1_macro,inf_ms
0,svm_tuned,classical,0.994487,0.993808,0.981786
1,knn_tuned,classical,0.994224,0.993499,0.901884


## Evaluate neural models

In [10]:
if X_test_neu is not None and cnn_best is not None:
    y_pred = cnn_best.predict(X_test_neu).argmax(axis=1)
    acc = accuracy_score(y_test, y_pred); f1 = f1_score(y_test, y_pred, average='macro')
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5,4)); sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=classes, yticklabels=classes)
    plt.title('CM — Custom CNN'); plt.tight_layout(); plt.savefig(FIG_DIR / 'cm_cnn.png', dpi=150); plt.close()
    results.append({'model': 'custom_cnn', 'type': 'neural', 'accuracy': acc, 'f1_macro': f1})
if X_test_neu is not None and yam_head is not None:
    # Use embeddings head evaluation from 08 (assumes X_test_neu aligned; if not, skip)
    # For reporting, we may instead load cached embeddings and eval there.
    pass
df_neu_res = pd.DataFrame([{k:v for k,v in r.items() if k!='report'} for r in results if r['type']=='neural'])
df_neu_res


""


## Consolidated comparison and export

In [11]:
df_all = pd.DataFrame([{k:v for k,v in r.items() if k!='report'} for r in results])
df_all.to_csv(RESULTS_DIR / 'final_model_comparison.csv', index=False)
plt.figure(figsize=(7,4))
sns.barplot(data=df_all, x='model', y='f1_macro', hue='type')
plt.title('Macro F1 by model'); plt.tight_layout(); plt.savefig(FIG_DIR / 'final_f1_bar.png', dpi=150); plt.close()
df_all


,model,type,accuracy,f1_macro,inf_ms
0,svm_tuned,classical,0.994487,0.993808,0.981786
1,knn_tuned,classical,0.994224,0.993499,0.901884


## Error analysis (top confusions)

In [12]:
# Identify misclassifications for the best model (by F1)
best = df_all.sort_values('f1_macro', ascending=False).iloc[0]['model']
if best in models_classical:
    y_pred = models_classical[best].predict(X_test_cls)
elif best == 'custom_cnn' and X_test_neu is not None and cnn_best is not None:
    y_pred = cnn_best.predict(X_test_neu).argmax(axis=1)
else:
    y_pred = None
if y_pred is not None:
    mis = np.where(y_pred != y_test)[0]
    print(f'Misclassified samples: {len(mis)}')
    if len(mis):
        # Show confusion pairs counts
        pairs = pd.DataFrame({'true': y_test[mis], 'pred': y_pred[mis]})
        counts = pairs.value_counts().reset_index(name='count')
        counts['true_lbl'] = counts['true'].map(lambda i: classes[i])
        counts['pred_lbl'] = counts['pred'].map(lambda i: classes[i])
        print(counts.head(10))
else:
    print('No predictions available for error analysis')


Misclassified samples: 21
   true  pred  count       true_lbl       pred_lbl
0     2     0     13          tools  angle_grinder
1     1     2      3     background          tools
2     0     1      2  angle_grinder     background
3     2     1      2          tools     background
4     1     0      1     background  angle_grinder


## Optional: TFLite accuracy check (neural)

In [13]:
# If TFLite models exist from 09, measure accuracy drop vs float32
def tflite_predict(tflite_path, X):
    if not Path(tflite_path).exists(): return None
    inter = tflite.Interpreter(model_path=str(tflite_path)); inter.allocate_tensors()
    inp = inter.get_input_details()[0]; out = inter.get_output_details()[0]
    preds = []
    for i in range(len(X)):
        inter.set_tensor(inp['index'], X[i:i+1].astype(np.float32))
        inter.invoke()
        preds.append(inter.get_tensor(out['index']))
    return np.vstack(preds)

f16 = PROJECT_ROOT / 'models' / 'tflite' / 'custom_cnn_f16.tflite'
i8  = PROJECT_ROOT / 'models' / 'tflite' / 'custom_cnn_int8.tflite'
if X_test_neu is not None and Path(f16).exists():
    p = tflite_predict(f16, X_test_neu)
    if p is not None:
        y_hat = p.argmax(axis=1)
        print('TFLite f16 Acc:', accuracy_score(y_test, y_hat))
if X_test_neu is not None and Path(i8).exists():
    p = tflite_predict(i8, X_test_neu)
    if p is not None:
        y_hat = p.argmax(axis=1)
        print('TFLite int8 Acc:', accuracy_score(y_test, y_hat))


## Export summary artifacts

In [14]:
# Save per-class metrics table (precision/recall/F1) for the best model
best = df_all.sort_values('f1_macro', ascending=False).iloc[0]['model']
if best in models_classical:
    rep = classification_report(y_test, models_classical[best].predict(X_test_cls), target_names=classes, output_dict=True)
elif best == 'custom_cnn' and X_test_neu is not None and cnn_best is not None:
    rep = classification_report(y_test, cnn_best.predict(X_test_neu).argmax(axis=1), target_names=classes, output_dict=True)
else:
    rep = None
if rep is not None:
    per_class = pd.DataFrame(rep).T
    per_class.to_csv(RESULTS_DIR / 'best_model_per_class_metrics.csv')
# Save a markdown-like text summary
summary = {
  'best_model': best,
  'models_evaluated': list(df_all['model'].unique()),
  'figures': [str(p) for p in (FIG_DIR.glob('cm_*.png'))],
}
with open(RESULTS_DIR / 'evaluation_summary.json', 'w') as f: json.dump(summary, f, indent=2)
print('Saved evaluation summary and per-class metrics.')


Saved evaluation summary and per-class metrics.
